# Citation Faithfulness: When NLI Lies About RAG Answers

**Andrea Trugenberger** &middot; SCIPER 357615 &middot; CS-552 Spring 2026 &middot;
Team Faithful RAG &middot; M2 Progress

---

## Question

When a language model gives an answer with a citation, how do we automatically check that the cited passage actually supports the claim? My contribution asks: **Is NLI good enough on its own to catch hallucinated citations, or does it miss claims that are topically related but factually wrong?**

The answer turns out to be no, NLI is not good enough. On a closed-book hallucination test across three LLMs, NLI labels 75% of made-up answers as faithful. An LLM judge correctly flags them. The gap is 55–61 percentage points across every model I tried.

## What I built

* **`evaluation/faithfulness/faithfulness_scorer.py`** — the verification pipeline. The full source is pasted inline below.
* **`scripts/run_faithfulness_experiment_full.py`** — the experiment driver. Loads answerable questions from `gold_qa.json`, asks each of three LLMs to answer closed-book, runs both verifiers on every generated claim against the gold passage.
* **Two write-ups** — `01_llm_judge_vs_nli.md` (n=1 pilot) and `02_full_results.json` (n=24 raw output that this notebook analyses).

## The pipeline in one cell

The cell below is the full contents of `faithfulness_scorer.py` from the merged main branch. It defines four functions:

* `extract_claims(answer)` — FActScore-style atomic claim decomposition via GPT-4o-mini, with a regex sentence-split fallback when no API key is set.
* `verify_claim_nli(claim, passage)` — zero-shot DeBERTa NLI; maps entailment to `SUPPORTED`, contradiction/neutral to `NOT_SUPPORTED`.
* `verify_claim_llm_judge(claim, passage)` — GPT-4o-mini prompted to require specific entities and terms in the claim to appear in the passage. Returns label, confidence, and a one-sentence reason. This is what catches the hallucinated-specifics failure mode NLI misses.
* `verify_claim(claim, known_source_ids, verifier)` — the four-way orchestrator. Handles `NO_CITATION` (claim has no citation), `FABRICATED_SOURCE` (cited paper not in corpus), and routes the rest to the chosen verifier backend.


In [8]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum


class VerificationLabel(str, Enum):
    SUPPORTED = "supported"
    NOT_SUPPORTED = "not_supported"
    FABRICATED_SOURCE = "fabricated_source"
    NO_CITATION = "no_citation"


@dataclass(frozen=True)
class Claim:
    text: str
    cited_source_id: str | None
    cited_passage: str | None


@dataclass(frozen=True)
class VerificationResult:
    claim: Claim
    label: VerificationLabel
    confidence: float
    explanation: str


def extract_claims(answer: str, model: str = "openai/gpt-4o-mini") -> list[str]:
    """
    Extract atomic factual claims from a generated RAG answer using an LLM.

    Falls back to rule-based sentence splitting if no API key is set.

    Args:
        answer: The full text of the generated RAG answer
        model: OpenRouter model spec (default: GPT-4o-mini, cheap and fast)

    Returns:
        A list of atomic claim strings, each a single verifiable proposition.
    """
    import os
    import json
    import urllib.request
    import re

    api_key = os.environ.get("OPENROUTER_API_KEY")

    # Fallback: rule-based splitting if no API key
    if not api_key:
        sentences = re.split(r"(?<=[.!?])\s+", answer.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 15]

    # LLM-based decomposition (FActScore-style)
    prompt = f"""Decompose the following answer into atomic factual claims.
Each claim must be ONE independently verifiable proposition.
If two facts are joined by "and", split them.
Ignore questions, opinions, and meta-commentary.
Return ONLY a JSON array of strings, no other text.

Answer:
\"\"\"{answer}\"\"\"

JSON array:"""

    body = json.dumps(
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0,
        }
    ).encode()

    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
    )

    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read())

    text = data["choices"][0]["message"]["content"].strip()

    # Strip markdown code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


def verify_claim_nli(
    claim: str,
    passage: str,
    nli_pipeline=None,
) -> tuple[VerificationLabel, float]:
    """
    Use NLI model to check if passage entails the claim.

    Args:
        claim: The atomic claim to verify
        passage: The cited passage that supposedly supports the claim
        nli_pipeline: Optional pre-loaded HuggingFace pipeline (for efficiency).
                      If None, loads the model on first call.

    Returns:
        Tuple of (VerificationLabel, confidence score)
    """
    from transformers import pipeline

    # Load model if not provided (allows reuse across many calls)
    if nli_pipeline is None:
        nli_pipeline = pipeline(
            "zero-shot-classification", model="cross-encoder/nli-deberta-v3-small"
        )

    # Run NLI: does the passage entail, contradict, or stay neutral to the claim?
    result = nli_pipeline(
        passage,
        candidate_labels=["entailment", "contradiction", "neutral"],
        hypothesis_template="This text means that {}.",
    )

    top_label = result["labels"][0]
    top_score = float(result["scores"][0])

    # Map NLI labels to our citation labels
    if top_label == "entailment" and top_score > 0.5:
        return VerificationLabel.SUPPORTED, round(top_score, 3)
    elif top_label == "contradiction":
        return VerificationLabel.NOT_SUPPORTED, round(top_score, 3)
    else:
        return VerificationLabel.NOT_SUPPORTED, round(top_score, 3)


def verify_claim_llm_judge(
    claim: str,
    passage: str,
    model: str = "openai/gpt-4o-mini",
) -> tuple[VerificationLabel, float, str]:
    """
    Use an LLM judge to check whether the passage SPECIFICALLY supports the claim.

    Catches a failure mode that NLI misses: claims that are topically related
    to the passage but state different specifics (hallucinated details).

    Returns
    -------
    (label, confidence, explanation)
    """
    import os
    import json
    import urllib.request
    import re

    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY required for LLM judge")

    prompt = f"""You are a strict citation verifier. Determine whether the PASSAGE specifically supports the CLAIM.

A claim is SUPPORTED only if the specific facts, terms, and entities in the claim appear in the passage.
A claim that is topically related but states different specifics is NOT_SUPPORTED.

PASSAGE:
\"\"\"{passage}\"\"\"

CLAIM:
\"\"\"{claim}\"\"\"

Return ONLY a JSON object with these fields:
{{"label": "supported" | "not_supported", "confidence": 0.0 to 1.0, "reason": "one sentence"}}"""

    body = json.dumps(
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0,
        }
    ).encode()

    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
    )

    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read())

    text = data["choices"][0]["message"]["content"].strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    parsed = json.loads(text)

    label = (
        VerificationLabel.SUPPORTED
        if parsed["label"] == "supported"
        else VerificationLabel.NOT_SUPPORTED
    )
    return label, float(parsed["confidence"]), parsed.get("reason", "")


def verify_claim(
    claim: Claim,
    known_source_ids: set[str],
    verifier: str = "nli",
    nli_pipeline=None,
    judge_model: str = "openai/gpt-4o-mini",
) -> VerificationResult:
    """
    Orchestrate the full four-way verification of a single claim.

    Decision tree:
        1. No cited_source_id          -> NO_CITATION
        2. cited_source_id not in corpus -> FABRICATED_SOURCE
        3. Otherwise                   -> run verifier on (claim, passage)
                                          -> SUPPORTED or NOT_SUPPORTED

    Args:
        claim: The Claim to verify.
        known_source_ids: Set of source IDs that exist in the retrieval corpus.
                          Used to detect fabricated citations.
        verifier: "nli" (fast, free) or "llm_judge" (slower, costs API calls,
                  catches hallucinated specifics).
        nli_pipeline: Optional pre-loaded HF pipeline for the NLI path.
        judge_model: OpenRouter model spec for the LLM-judge path.

    Returns:
        A VerificationResult with label, confidence, and a short explanation.
    """
    # Case 1: the model didn't cite anything
    if claim.cited_source_id is None:
        return VerificationResult(
            claim=claim,
            label=VerificationLabel.NO_CITATION,
            confidence=1.0,
            explanation="Claim was generated without any citation.",
        )

    # Case 2: the model cited a source that doesn't exist in our corpus
    if claim.cited_source_id not in known_source_ids:
        return VerificationResult(
            claim=claim,
            label=VerificationLabel.FABRICATED_SOURCE,
            confidence=1.0,
            explanation=(f"Cited source '{claim.cited_source_id}' is not in the corpus."),
        )

    # Case 3: real source, real passage — does it actually support the claim?
    # Guard against the edge case where the source exists but no passage was
    # attached to the Claim (shouldn't happen in practice, but be defensive).
    if claim.cited_passage is None:
        return VerificationResult(
            claim=claim,
            label=VerificationLabel.NOT_SUPPORTED,
            confidence=1.0,
            explanation="Source exists but no passage was provided for verification.",
        )

    if verifier == "nli":
        label, confidence = verify_claim_nli(
            claim.text, claim.cited_passage, nli_pipeline=nli_pipeline
        )
        explanation = f"NLI verifier returned {label.value} (score={confidence})."
        return VerificationResult(
            claim=claim,
            label=label,
            confidence=confidence,
            explanation=explanation,
        )

    if verifier == "llm_judge":
        label, confidence, reason = verify_claim_llm_judge(
            claim.text, claim.cited_passage, model=judge_model
        )
        return VerificationResult(
            claim=claim,
            label=label,
            confidence=confidence,
            explanation=f"LLM judge: {reason}",
        )

    raise ValueError(f"Unknown verifier: {verifier!r}. Use 'nli' or 'llm_judge'.")


def compute_faithfulness_score(results: list[VerificationResult]) -> dict:
    """Aggregate faithfulness metrics across all claims."""
    if not results:
        return {"faithfulness": 0.0, "total_claims": 0}

    total = len(results)
    supported = sum(1 for r in results if r.label == VerificationLabel.SUPPORTED)
    fabricated = sum(1 for r in results if r.label == VerificationLabel.FABRICATED_SOURCE)

    return {
        "faithfulness": supported / total,
        "supported_ratio": supported / total,
        "not_supported_ratio": sum(1 for r in results if r.label == VerificationLabel.NOT_SUPPORTED)
        / total,
        "fabricated_ratio": fabricated / total,
        "no_citation_ratio": sum(1 for r in results if r.label == VerificationLabel.NO_CITATION)
        / total,
        "total_claims": total,
    }


## Design choice: a four-way label, not a binary one

The proposal calls for a faithfulness check, which sounds binary. In practice an LLM answer can fail to be faithful in four different ways. The first two cases (`NO_CITATION`, `FABRICATED_SOURCE`) are decided **without running any model** — they fall out of structural checks. Only the last two need a verifier.

## Design choice: two verifiers, not one

NLI was the obvious starting point because it is what the literature uses (ALCE; FActScore). Early in the project I ran a simple test: a passage that says *"Shapiro identifies two transaction costs: complements and hold-up"* and a claim that says *"Shapiro identifies three transaction costs including a licensing problem."* NLI labelled it `supported` with confidence 0.984 — exactly the same confidence as a correct claim.

NLI measures *topical entailment*. Both texts are about Shapiro, transaction costs, patents. NLI sees enough overlap to call it entailment. It cannot see that the *specifics* are wrong. So I added the LLM-judge head — cheap-but-blind versus expensive-but-careful.

## Experimental setup

For each of the 24 answerable gold questions, three LLMs answer closed-book — no retrieval, no context. This forces the model to answer from training data alone. Then each generated answer is decomposed into atomic claims, and each claim runs through both verifiers, with the **gold passage** from `gold_qa.json` as the cited passage.

Closed-book is deliberately the hardest faithfulness setting: any faithfulness is by coincidence rather than retrieval. This is where the verifier gap is most visible.


In [9]:
"""Set up paths so we can load results."""
import json
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Repo root not found.")
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

results_path = repo_root / "evaluation/faithfulness/results/02_full_results.json"
all_results = json.loads(results_path.read_text())

n_runs = len(all_results)
models = sorted({r["model"] for r in all_results if "model" in r})
print(f"Total (question, model) runs: {n_runs}")
print(f"Models compared: {models}")
print(f"Questions covered: {len({r['question_id'] for r in all_results})}")


Total (question, model) runs: 72
Models compared: ['anthropic/claude-3.5-haiku', 'deepseek/deepseek-chat', 'openai/gpt-4o-mini']
Questions covered: 24


## Headline numbers

In [10]:
"""Aggregate per-model faithfulness across all questions."""
import pandas as pd

rows = []
for model in models:
    model_rows = [r for r in all_results if r.get("model") == model and "error" not in r]
    n = len(model_rows)
    nli_avg = sum(r["nli_faithfulness"] for r in model_rows) / n
    judge_avg = sum(r["judge_faithfulness"] for r in model_rows) / n
    rows.append({
        "model": model,
        "n_questions": n,
        "total_claims": sum(r["n_claims"] for r in model_rows),
        "NLI faithfulness": round(nli_avg, 3),
        "LLM-judge faithfulness": round(judge_avg, 3),
        "gap (NLI - judge)": round(nli_avg - judge_avg, 3),
    })

headline_df = pd.DataFrame(rows).set_index("model")
headline_df


,n_questions,total_claims,NLI faithfulness,LLM-judge faithfulness,gap (NLI - judge)
model,,,,,
anthropic/claude-3.5-haiku,24,174,0.75,0.173,0.577
deepseek/deepseek-chat,24,178,0.75,0.198,0.552
openai/gpt-4o-mini,24,183,0.75,0.138,0.612


**Reading the table.**

* **All three generators get 75.0% NLI faithfulness.** NLI rubber-stamps anything topically related as `supported`. It cannot tell the three models apart because it cannot see what each got specifically wrong.
* **The LLM judge separates them.** GPT-4o-mini 13.8%, Claude 3.5 Haiku 17.3%, DeepSeek-chat 19.8%.
* **The gap is 55–61 percentage points and is stable across providers.** This is the headline finding: a systematic failure of NLI as a citation-faithfulness gate.


In [11]:
"""Case study: q004 — the two transaction costs in Shapiro 2001."""
q004 = [r for r in all_results if r.get("question_id") == "q004"]
for r in q004:
    if "error" in r:
        continue
    print(f"\n=== {r['model']} ===")
    print(f"Answer: {r['answer'][:300]}...")
    print(f"NLI:   {r['nli_faithfulness']:.1%}    Judge: {r['judge_faithfulness']:.1%}")
    print("Per-claim (first 3):")
    for c in r["claims"][:3]:
        print(f"  NLI={c['nli_label']:<14}  Judge={c['judge_label']:<14}  {c['claim'][:80]}")



=== openai/gpt-4o-mini ===
Answer: According to Shapiro (2001), the two main types of transaction costs imposed by overlapping patents on innovation are negotiation costs and enforcement costs. Negotiation costs arise from the need to obtain licenses from multiple patent holders, while enforcement costs relate to the challenges and e...
NLI:   100.0%    Judge: 0.0%
Per-claim (first 3):
  NLI=supported       Judge=not_supported   According to Shapiro (2001), there are two main types of transaction costs impos
  NLI=supported       Judge=not_supported   The two main types of transaction costs are negotiation costs and enforcement co
  NLI=supported       Judge=not_supported   Negotiation costs arise from the need to obtain licenses from multiple patent ho

=== anthropic/claude-3.5-haiku ===
Answer: According to Shapiro (2001), the two main types of transaction costs imposed by overlapping patents are: (1) search costs, which arise from the difficulty and expense of identifying and locat

**What happened on q004.** Gold answer: complements problem and hold-up problem. None of the three models produced these names. GPT-4o-mini said "negotiation costs" and "enforcement costs"; Claude said "search costs" and "licensing costs"; DeepSeek said "search costs" and "bargaining costs". NLI labelled all of them supported. The judge correctly labelled them not-supported.

## q017: invented numbers


In [12]:
"""Case study: q017 — annual US patent issuance growth 1983-2010."""
q017 = [r for r in all_results if r.get("question_id") == "q017"]
for r in q017:
    if "error" in r:
        continue
    print(f"\n=== {r['model']} ===")
    print(f"Answer: {r['answer'][:300]}...")
    print(f"NLI:   {r['nli_faithfulness']:.1%}    Judge: {r['judge_faithfulness']:.1%}")



=== openai/gpt-4o-mini ===
Answer: According to Boldrin and Levine (2013), annual US patent issuance grew from approximately 90,000 patents in 1983 to about 200,000 patents in 2010, indicating a significant increase over that period. This growth reflects a more than twofold increase in the number of patents issued annually....
NLI:   100.0%    Judge: 25.0%

=== anthropic/claude-3.5-haiku ===
Answer: According to Boldrin and Levine (2013), annual US patent issuance grew from approximately 50,000 patents per year in 1983 to around 250,000 patents per year in 2010, representing a five-fold increase over this 27-year period. This substantial growth in patent issuance occurred across various technol...
NLI:   100.0%    Judge: 14.3%

=== deepseek/deepseek-chat ===
Answer: According to Boldrin and Levine (2013), annual U.S. patent issuance grew significantly between 1983 and 2010, increasing from approximately 62,000 patents to over 240,000 patents. This represents nearly a fourfold growth o

**What happened on q017.** Three LLMs, three different made-up numbers for the same factual question. NLI gave all three 100% because the topic matches. The judge correctly catches the specific number mismatches.

## q018: a moment of honesty


In [13]:
"""Case study: q018 — Claude's honest refusal."""
q018 = [r for r in all_results if r.get("question_id") == "q018"]
for r in q018:
    if "error" in r or "claude" not in r["model"]:
        continue
    print(f"=== {r['model']} ===")
    print(f"Answer: {r['answer']}")
    print(f"NLI:   {r['nli_faithfulness']:.1%}    Judge: {r['judge_faithfulness']:.1%}")


=== anthropic/claude-3.5-haiku ===
Answer: I apologize, but I cannot confidently answer this question without having direct access to Moser's 2013 research. Without reading the specific paper, I cannot accurately represent the author's conclusions about patent laws and innovation. To provide a reliable response, I would need to review the original source material.
NLI:   100.0%    Judge: 0.0%


**What happened on q018.** Claude said *"I cannot confidently answer this without direct access to Moser's paper."* The right answer for a closed-book question. GPT and DeepSeek both hallucinated specifics on the same question.

## Limitations

1. **Closed-book is the worst case for the generators.** With retrieval the gap would shrink. The 55–61 pp gap is an upper bound on verifier disagreement.
2. **The gold dataset has no `not_supports` claims.** Every gold claim is true with a real supporting passage. The experiment measures the verifier gap on potentially hallucinated generated claims, but cannot measure verifier precision on hand-crafted false claims. Faruk's adversarial controls (q900–q905) are starting to fix this.
3. **The `FABRICATED_SOURCE` path is unused here.** The orchestrator handles this case (tested separately on a fake paper id), but empirical evaluation is M3 work.

## M3 plan

* Re-run the cross-LLM experiment **with retrieval** using Elie's best retriever (E5-large coarse rerank).
* Run both verifiers on the **adversarial control set** to measure precision on intentionally false claims.
* Try the larger NLI model originally specified in the proposal to see how much of the gap is small-model capacity versus NLI as a technique.
